<a href="https://colab.research.google.com/github/MarquiseRosier/pi05-run/blob/feature/marquise-transcoder-feature-inspection/notebooks/pi05_transcoder_counterfactual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Pi0.5 Transcoder Counterfactual Probe

Standalone notebook for one study: **which transcoder features respond to a
single object's appearance, and does a traced circuit carry that response?**

It recolors one object at a frozen simulator state, renders the scene twice,
and pushes both observations through the policy with the same flow-matching
noise. Physics is never stepped between the renders, so any activation
difference is attributable to that object's pixels.

## What each stage decides

| Stage | Question | Decided on |
|---|---|---|
| Probe | H1: do features respond to the referent bowl more than to the identical non-referent bowl, once footprint is accounted for? Scaled against a positive control (the prompt swap) and checked across scale-free summaries. | `decision_metrics.json` → `h1` |
| Probe | H2: does that selectivity follow the language, or the object's position? | `decision_metrics.json` → `h2` |
| Report | Which feature to trace | `nominated_targets.json` (one rule, applied once) |
| Trace + Validate | H3: do the traced parents carry the perturbed property? | `circuit/counterfactual_validation/validation.json` → `h3_verdict` |

Every decision rule is written next to its verdict in those files.

## Why this is separate from the main notebook

The full simulation notebook mounts Drive caches, downloads the LIBERO dataset
and runs closed-loop evals. None of that is needed for the probe. It uses
`make_env`, not `make_dataset`, so there is **no dataset download** for the
probe stage, and it runs forward passes rather than rollouts. Only the circuit
trace needs dataset rows.

Drive is mounted for two reasons: to read the transcoder checkpoint, and to
keep every run. Colab's disk is wiped when the runtime ends, so each stage
copies its run directory to `DRIVE_ROOT/outputs/` (probe runs under
`probes/counterfactual/<stamp>/`, with the circuit and its validation inside;
discovery artefacts under `features/pi05_libero/counterfactual-<stamp>/`).

## Order

Run the cells top to bottom. Nothing needs to be filled in: the probe reads the
task's BDDL and perturbs its `obj_of_interest`, using another instance of the
same object type as a matched control.

1. Controls
2. Runtime check and Drive mount
3. Install
4. Clone repo
5. HF token and LIBERO assets
6. Resolve checkpoint
7. Run the probe (H1, H2, nomination)
8. Trace and validate the circuit (H3)


In [ ]:
# @title Controls

# Repo.
REPO_URL = "https://github.com/MarquiseRosier/pi05-run.git"  # @param {type:"string"}
REPO_BRANCH = "feature/marquise-transcoder-feature-inspection"  # @param {type:"string"}

# Drive is used only to read the transcoder checkpoint.
DRIVE_ROOT = "/content/drive/MyDrive/groot-run-shared-programmer908"  # @param {type:"string"}
TRANSCODER_DRIVE_PATH = "transcoders/pi05_libero/allframes_80-10-10_epoch1_b8_exp16_latest_lambda1e-4/step_027233.pt"  # @param {type:"string"}

# Scene.
POLICY_PATH = "lerobot/pi05_libero_finetuned"  # @param {type:"string"}
SUITE = "libero_spatial"  # @param ["libero_spatial", "libero_object", "libero_goal", "libero_10"]
SEED = 1000  # @param {type:"integer"}

# Extra seeds change where the objects start, so they are genuinely different
# scenes within a task rather than more of the same one. They tighten each
# task's mean; only more tasks widen the cluster count the interval rests on.
SEEDS = "1000,1001"  # @param {type:"string"}

# Every libero_spatial task instantiates two identical akita bowls, so the
# matched placebo works on all ten. The task is the resampling unit, so ten of
# them is what makes an interval possible; a single task cannot support one.
TASK_IDS = "0-9"  # @param {type:"string"}

# Perturbation. Both default to the task's own objects: TARGET becomes the
# BDDL's first obj_of_interest, PLACEBO_TARGET another instance of the same
# object type. Leave them blank unless you want to override.
TARGET = ""  # @param {type:"string"}
PLACEBO_TARGET = ""  # @param {type:"string"}
LIST_OBJECTS_ONLY = False  # @param {type:"boolean"}
PERTURBATION = "blend"  # @param ["blend", "set", "hue"]
COLOR = "1.0,0.2,0.1"  # @param {type:"string"}
DOSE = "0.5,1.0"  # @param {type:"string"}

# Replication. Cells = STATES x NOISE_SAMPLES x doses under the task prompt;
# these are the replicates behind the error bound on H1. Noise draws are seeded
# from SEED, so the run is reproducible bit for bit.
# States are the largest source of spread: in the two-state pilot the adjusted
# selectivity was ~1.3 at the reset pose and ~11.8 five actions later. Eight
# states per task samples that curve instead of guessing from its endpoints.
STATES = 12  # @param {type:"integer"}
STATE_STRIDE = 4  # @param {type:"integer"}
NOISE_SAMPLES = 2  # @param {type:"integer"}

# Nomination rule for the trace target (applied once, by the report, and read
# by the trace cell). A candidate must fire in at least MIN_TRACE_CONSISTENCY
# of the task prompt's cells, sit at layer >= MIN_TRACE_LAYER so it has parents
# to find, and respond at most 1/MIN_SELECTIVITY as much to the placebo.
MIN_TRACE_CONSISTENCY = 0.25  # @param {type:"number"}
MIN_TRACE_LAYER = 4  # @param {type:"integer"}
MIN_SELECTIVITY = 2.0  # @param {type:"number"}

# Circuit tracing. The tracer needs feature-discovery artifacts, which come
# from the LIBERO dataset rather than the live env, so this stage downloads the
# dataset. Scoped small: we only need top-K for the nominated feature.
RUN_CIRCUIT_TRACE = True  # @param {type:"boolean"}
DISCOVERY_EPISODES = "0,1,2,3,4"  # @param {type:"string"}
DISCOVERY_MAX_BATCHES = 40  # @param {type:"integer"}
TRACE_PARENTS_PER_NODE = 2  # @param {type:"integer"}
TRACE_MAX_DEPTH = 3  # @param {type:"integer"}
TRACE_MAX_NODES = 60  # @param {type:"integer"}
VALIDATION_RANDOM_DRAWS = 2000  # @param {type:"integer"}

# H3 as a rate rather than an anecdote: trace and audit this many nominated
# targets, pooled across every task, so the rate is over the method and not
# over one scene. One target can only ever produce a single falsification.
# With k of N corroborated the verdict is read off a Clopper-Pearson interval,
# so N is what decides how much a null result can exclude: 0/5 bounds the rate
# below 45%, 0/10 below 26%, 0/20 below 14%. Twenty is the settling number.
TRACE_N_TARGETS = 20  # @param {type:"integer"}
# Sources for the frontier. The pilot used all-earlier from layer 5, and 45 of
# 60 expansions collapsed into layer 1. previous-layer is the control for that.
TRACE_SOURCE_POLICY = "previous-layer"  # @param ["previous-layer", "all-earlier"]
# Calibrate what the audit can detect before trusting what it did not.
RUN_AUDIT_CALIBRATION = True  # @param {type:"boolean"}

# Every run directory (probe, circuit, feature discovery) is copied to
# DRIVE_ROOT/outputs/ after each stage. Colab's disk is wiped when the runtime
# ends; the delta store, verdicts and provenance would go with it.
SAVE_RUNS_TO_DRIVE = True  # @param {type:"boolean"}

MIN_GPU_MEMORY_GB = 20  # @param {type:"integer"}

def _parse_ids(spec):
    out = []
    for part in str(spec).split(","):
        part = part.strip()
        if not part:
            continue
        if "-" in part:
            lo, hi = part.split("-"); out.extend(range(int(lo), int(hi) + 1))
        else:
            out.append(int(part))
    return out


TASK_ID_LIST = _parse_ids(TASK_IDS)
SEED_LIST = _parse_ids(SEEDS) or [SEED]
_doses = len([d for d in DOSE.split(",") if d.strip()])
_cells = STATES * NOISE_SAMPLES * _doses * len(SEED_LIST)
_runs = len(TASK_ID_LIST) * len(SEED_LIST)
_forwards = _runs * STATES * NOISE_SAMPLES * 2 * (2 + 2 * _doses)
print(f"Suite: {SUITE} | tasks: {TASK_ID_LIST} | seeds: {SEED_LIST}")
print(f"Cells per condition per task: {_cells}  "
      f"({len(SEED_LIST)} seeds x {STATES} states x {NOISE_SAMPLES} draws x {_doses} doses)")
print(f"Task clusters: {len(TASK_ID_LIST)}  (the unit the confidence interval resamples)")
print(f"Probe invocations: {_runs};  forward passes: about {_forwards:,}")
print(f"Rough A100 time: {_runs * 110 / 60 + _forwards / 60:.0f} min "
      f"(~{_runs * 110 / 60:.0f} min of it policy loading)")
print(f"Delta store on Drive: about {_runs * STATES * NOISE_SAMPLES * 2 * 3 / 1024:.1f} GB")
print("Target:", repr(TARGET) or "(each task's own object of interest)")

if RUN_CIRCUIT_TRACE:
    # A trace is 60 frontier expansions, each a VJP through the action expert.
    # The pilot took 1312 s at these settings, which is what this extrapolates.
    _trace_min = TRACE_N_TARGETS * 1312 / 60
    _bound = 1 - 0.05 ** (1 / max(TRACE_N_TARGETS, 1))
    print(f"\nH3: {TRACE_N_TARGETS} targets, about {_trace_min / 60:.1f} h of tracing "
          f"plus ~25 min of discovery.")
    print(f"    If none is corroborated, the rate is bounded below {_bound:.1%} "
          f"(Clopper-Pearson, one-sided 95%).")
    if _bound > 0.5:
        print("    That bound is above 50%, so a null result could not even exclude "
              "'works half the time'. Raise TRACE_N_TARGETS.")


In [ ]:
# @title Validate Runtime And Mount Drive

import os
import shutil
import subprocess
import time
from pathlib import Path

from google.colab import drive


def gpu_info():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, check=True,
        ).stdout.strip().splitlines()[0]
        name, mem = [part.strip() for part in out.split(",")]
        return name, int(mem) // 1024
    except Exception:
        return None, 0


gpu_name, gpu_mem_gb = gpu_info()
if gpu_name is None:
    raise RuntimeError("No GPU. Runtime > Change runtime type > L4.")
print(f"GPU: {gpu_name} (~{gpu_mem_gb} GiB)")
if gpu_mem_gb < int(MIN_GPU_MEMORY_GB):
    raise RuntimeError(
        f"{gpu_name} has ~{gpu_mem_gb} GiB; Pi0.5 needs >= {MIN_GPU_MEMORY_GB} GiB. Use an L4 or A100."
    )


def mount_drive_with_retry(mountpoint: str = "/content/drive", attempts: int = 3) -> None:
    """Mount Drive, retrying because `mount failed` is usually transient."""
    if Path(mountpoint, "MyDrive").exists():
        print(f"Drive already mounted at {mountpoint}", flush=True)
        return
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            drive.mount(mountpoint, force_remount=attempt > 1)
            print(f"Drive mounted at {mountpoint} (attempt {attempt})", flush=True)
            return
        except Exception as exc:
            last_error = exc
            print(f"Drive mount attempt {attempt}/{attempts} failed: {exc}", flush=True)
            if attempt < attempts:
                delay = 5 * attempt
                print(f"  retrying in {delay}s", flush=True)
                time.sleep(delay)
    raise RuntimeError(
        f"Could not mount Google Drive after {attempts} attempts (last error: {last_error}).\n"
        "Drive is needed here only to read the transcoder checkpoint.\n"
        "Most likely causes, in order:\n"
        "  1. Other Colab sessions hold Drive mounts. Runtime > Manage sessions, terminate "
        "the ones you are not using, then Runtime > Restart session and rerun.\n"
        "  2. The authorization popup was blocked. Allow popups and third-party cookies.\n"
        "  3. A transient Drive outage. Restart the runtime and retry in a few minutes."
    ) from last_error


mount_drive_with_retry()
DRIVE_ROOT = Path(DRIVE_ROOT)
print("Drive root:", DRIVE_ROOT, "exists:", DRIVE_ROOT.exists())

# Everything this notebook measures is written under DRIVE_ROOT/outputs, because
# Colab's disk is wiped when the runtime ends. If the folder is not actually
# there, copytree would happily create a fresh unshared directory and the run
# would look saved while being invisible to anyone else. Fail here instead.
DRIVE_OUTPUTS = DRIVE_ROOT / "outputs"
if SAVE_RUNS_TO_DRIVE:
    if not DRIVE_ROOT.exists():
        raise RuntimeError(
            f"{DRIVE_ROOT} does not exist under the mounted Drive.\n"
            "Check DRIVE_ROOT in Controls against the shared folder, and make sure the "
            "folder is added to *your* My Drive (Shared with me is not mounted).\n"
            "Set SAVE_RUNS_TO_DRIVE = False only if you accept losing the run."
        )
    DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)
    probe = DRIVE_OUTPUTS / ".write_test"
    probe.write_text("ok")          # a read-only share fails here, not four hours in
    assert probe.read_text() == "ok"
    probe.unlink()
    free_gb = shutil.disk_usage(DRIVE_ROOT).free / 1024**3
    print(f"Drive writable. Free on the mount: {free_gb:.1f} GB")
    print(f"Everything this notebook produces lands under {DRIVE_OUTPUTS}")
else:
    print("SAVE_RUNS_TO_DRIVE is off: this run will be lost when the runtime ends.")


In [ ]:
# @title Install Runtime

import os
import subprocess
from pathlib import Path

VENV = Path("/content/lerobot-venv")
PYTHON = VENV / "bin/python"
UV_BIN_DIR = Path("/content/uv-bin")
UV = str(UV_BIN_DIR / "uv")


def run(cmd, *, env=None):
    cmd = list(map(str, cmd))
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, env=env, check=True)


if PYTHON.exists():
    print("venv already present; skipping install. Delete /content/lerobot-venv to force a rebuild.")
else:
    apt_packages = [
        "build-essential", "cmake", "curl", "ffmpeg", "git", "pkg-config",
        "libegl1", "libgl1", "libglib2.0-0", "libglvnd0", "libglx0", "libopengl0",
        "libosmesa6-dev", "libsm6", "libxext6", "libxrender1",
    ]
    apt_env = os.environ.copy()
    apt_env["DEBIAN_FRONTEND"] = "noninteractive"
    run(["apt-get", "update", "-qq"], env=apt_env)
    run(["apt-get", "install", "-y", "-qq", *apt_packages], env=apt_env)

    UV_BIN_DIR.mkdir(parents=True, exist_ok=True)
    run(["curl", "-LsSf", "https://astral.sh/uv/install.sh", "-o", "/tmp/install-uv.sh"])
    uv_env = os.environ.copy()
    uv_env["UV_INSTALL_DIR"] = str(UV_BIN_DIR)
    run(["sh", "/tmp/install-uv.sh"], env=uv_env)
    os.environ["PATH"] = f"{UV_BIN_DIR}:" + os.environ["PATH"]

    run([UV, "python", "install", "3.12"])
    run([UV, "venv", "--clear", str(VENV), "--python", "3.12"])
    # `evaluation` pulls the LIBERO env; no dataset extras are needed for this probe.
    run([
        UV, "pip", "install", "--python", str(PYTHON), "--torch-backend", "cu128",
        "lerobot[evaluation,libero,pi]", "hf-transfer", "opencv-python", "numpy",
    ])

os.environ["PATH"] = f"{VENV / 'bin'}:{UV_BIN_DIR}:" + os.environ["PATH"]
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
run([str(PYTHON), "-c",
     "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"])


In [ ]:
# @title Clone Or Update Repo

import subprocess
from pathlib import Path

LOCAL_REPO = Path("/content/pi05-run")
if LOCAL_REPO.exists():
    subprocess.run(["git", "-C", str(LOCAL_REPO), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(LOCAL_REPO), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(LOCAL_REPO), "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(LOCAL_REPO)], check=True)
subprocess.run(["git", "-C", str(LOCAL_REPO), "log", "-1", "--oneline"], check=True)


In [ ]:
# @title HF Token And LIBERO Assets

import os
from pathlib import Path

token = os.environ.get("HF_TOKEN", "")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN") or ""
    except Exception:
        token = ""
if not token:
    secret_file = DRIVE_ROOT / "secrets/HF_TOKEN.txt"
    if secret_file.exists():
        token = secret_file.read_text().strip()
if not token:
    raise RuntimeError(
        "No Hugging Face token. Add HF_TOKEN as a Colab secret (key icon in the sidebar), "
        f"or place it at {DRIVE_ROOT / 'secrets/HF_TOKEN.txt'}. "
        f"{POLICY_PATH} is gated, so the download needs it."
    )
os.environ["HF_TOKEN"] = token
print("HF token loaded.")

# LIBERO ships without its scene assets; fetch them into the installed package.
assets_code = r"""
import shutil, site, os
from pathlib import Path
from huggingface_hub import snapshot_download

roots = [Path(p) for p in site.getsitepackages()]
user_site = site.getusersitepackages()
if user_site:
    roots.append(Path(user_site))
libero_root = next((r / "libero" / "libero" for r in roots if (r / "libero" / "libero").exists()), None)
if libero_root is None:
    raise RuntimeError("Installed LIBERO package not found")
assets_dir = libero_root / "assets"
required = assets_dir / "scenes" / "libero_tabletop_base_style.xml"
if required.exists():
    print("LIBERO assets already present:", required)
else:
    snap = Path(snapshot_download(repo_id="lerobot/libero-assets", repo_type="dataset",
                                  token=os.environ.get("HF_TOKEN") or None))
    assets_dir.mkdir(parents=True, exist_ok=True)
    for child in snap.iterdir():
        if child.name == ".gitattributes":
            continue
        target = assets_dir / child.name
        if child.is_dir():
            shutil.copytree(child, target, dirs_exist_ok=True)
        else:
            shutil.copy2(child, target)
    if not required.exists():
        raise FileNotFoundError(f"LIBERO asset install failed; missing {required}")
    print("LIBERO assets installed:", required)
"""
run([str(PYTHON), "-c", assets_code])


In [ ]:
# @title Resolve Transcoder Checkpoint

import shutil
import time
from pathlib import Path

candidate = Path(TRANSCODER_DRIVE_PATH)
if not candidate.is_absolute():
    candidate = DRIVE_ROOT / candidate
if not candidate.exists():
    raise FileNotFoundError(
        f"Transcoder checkpoint not found: {candidate}\n"
        "Check TRANSCODER_DRIVE_PATH in Controls against the shared Drive folder."
    )

# Copy off Drive once. Reading a multi-GiB checkpoint repeatedly over the Drive
# FUSE mount is far slower than a local read, and it is the checkpoint load that
# the probe does on every run.
LOCAL_CHECKPOINT = Path("/content/checkpoints") / candidate.name
LOCAL_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if LOCAL_CHECKPOINT.exists() and LOCAL_CHECKPOINT.stat().st_size == candidate.stat().st_size:
    print("Local copy already present:", LOCAL_CHECKPOINT)
else:
    print(f"Copying {candidate.stat().st_size / 1024**3:.2f} GiB off Drive...", flush=True)
    started = time.time()
    shutil.copy2(candidate, LOCAL_CHECKPOINT)
    print(f"  done in {time.time() - started:.0f}s", flush=True)

TRANSCODER_CHECKPOINT = str(LOCAL_CHECKPOINT)
print("Checkpoint:", TRANSCODER_CHECKPOINT)
print("Size:", f"{LOCAL_CHECKPOINT.stat().st_size / 1024**3:.2f} GiB")


## Run The Probe

Just run it. The probe reads the task's BDDL and picks its own objects:

* **target** -- the task's first `obj_of_interest`. For `libero_spatial` task 0
  that is `akita_black_bowl_1`, the bowl the prompt refers to.
* **placebo** -- another instance of the same object type, here
  `akita_black_bowl_2`. Same mesh, same colour, same size, differing only in
  position and task relevance, which makes it a tightly matched control.

What to read in the output, in order:

1. `All keys loaded successfully!` -- the policy load. The run aborts otherwise.
2. `re-render liveness` -- the recolour moved pixels and the revert restored them.
3. `null control ... latent L2 mean=0` -- the nondeterminism floor. Any other
   value invalidates the paired comparison and the H1 verdict says so.
4. `--- decision metrics ---` -- the H1 and H2 tables, verdicts and rules. For H2
   read `referent check` first: if the action under the sibling prompt is still
   anchored on the target bowl, the swap moved words but not the referent and
   H2 is untestable on this scene, whatever `g` says.
5. `Nominated targets` -- the one rule for what gets traced.

Override `TARGET` / `PLACEBO_TARGET` in Controls only if you want something
else, or tick `LIST_OBJECTS_ONLY` to just inspect the scene.


In [ ]:
# @title Run Counterfactual Probe

import json
import os
import subprocess
import sys
import time
from pathlib import Path
from IPython.display import display, HTML, Markdown, Image as IPyImage

batch_dir = LOCAL_REPO / "outputs/probes/counterfactual" / time.strftime("%Y%m%d-%H%M%S", time.gmtime())
batch_dir.mkdir(parents=True, exist_ok=True)


import shutil

DRIVE_BATCH = (DRIVE_ROOT / batch_dir.relative_to(LOCAL_REPO)) if SAVE_RUNS_TO_DRIVE else None


def _tree_stats(root: Path) -> tuple[int, int]:
    files = [p for p in root.rglob("*") if p.is_file()]
    return len(files), sum(p.stat().st_size for p in files)


def save_to_drive(local_dir: Path, *, required: bool = True) -> Path | None:
    """Mirror one directory under DRIVE_ROOT/outputs/, preserving its path.

    Colab's disk does not survive the runtime, and neither does a disconnect
    four hours into a twenty-run loop. Every artefact the paper's tables are
    filled from (the delta store, decision_metrics.json, validation.json,
    provenance.json, the traced graph, the audit calibration) lives in these
    directories, so each is copied the moment it is complete rather than at the
    end of the notebook.

    The copy is verified. A Drive FUSE mount that has run out of quota returns
    success on write and leaves a short file, so comparing the byte count is
    the only way to know the data is really there.
    """
    if not SAVE_RUNS_TO_DRIVE:
        return None
    local_dir = Path(local_dir)
    if not local_dir.exists():
        return None
    rel = local_dir.relative_to(LOCAL_REPO)
    target = DRIVE_ROOT / rel
    src_files, src_bytes = _tree_stats(local_dir)
    try:
        shutil.copytree(local_dir, target, dirs_exist_ok=True)
        dst_files, dst_bytes = _tree_stats(target)
        short = src_bytes - dst_bytes
        if dst_files < src_files or short > 0:
            raise IOError(
                f"copy is short by {src_files - dst_files} files / {short / 1024**2:.1f} MB "
                f"(Drive quota or a dropped mount)"
            )
    except Exception as exc:
        message = f"!! FAILED to save {rel} to Drive: {exc}"
        if required:
            raise RuntimeError(
                message + f"\n   The data is still on local disk at {local_dir}. "
                "Free space or remount, then re-run this cell's save."
            ) from exc
        print(message, flush=True)
        return None
    print(f"saved {rel} -> {target}  ({dst_files} files, {dst_bytes / 1024**2:.0f} MB)", flush=True)
    return target


def write_manifest(**extra) -> None:
    """An index of the batch, written into Drive beside the runs.

    Further analysis should be able to start from the Drive folder alone,
    without the notebook or the Colab session that produced it.
    """
    if not SAVE_RUNS_TO_DRIVE:
        return
    manifest = {
        "batch": batch_dir.name,
        "written_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "drive_root": str(DRIVE_ROOT),
        "config": {
            "suite": SUITE, "task_ids": TASK_ID_LIST, "seeds": SEED_LIST,
            "states": STATES, "state_stride": STATE_STRIDE, "noise_samples": NOISE_SAMPLES,
            "perturbation": PERTURBATION, "color": COLOR, "dose": DOSE,
            "policy_path": POLICY_PATH, "transcoder_checkpoint": TRANSCODER_DRIVE_PATH,
            "trace_n_targets": TRACE_N_TARGETS, "trace_source_policy": TRACE_SOURCE_POLICY,
        },
        "runs": [], "failures": [{"task_id": t, "seed": s, "returncode": rc}
                                 for t, s, rc in failures],
        **extra,
        "reading": {
            "per task": "task*/decision_metrics.json -> h1, h2",
            "pooled over tasks": "aggregate/aggregate.json -> summary",
            "delta store": "task*/latents/ (signed per-feature deltas, npz blocks)",
            "H3 rate": "h3/h3_rate.json -> summary",
            "audit sensitivity": "h3/calibration_*/audit_calibration.json -> summary",
        },
    }
    for run_dir in run_dirs:
        entry = {"dir": run_dir.name}
        for name, key in (("decision_metrics.json", "verdicts"),
                          ("provenance.json", "provenance"),
                          ("nominated_targets.json", "nominated")):
            path = run_dir / name
            if not path.exists():
                continue
            data = json.loads(path.read_text())
            if key == "verdicts":
                entry["h1"] = data["h1"]["verdict"]
                entry["h2"] = data["h2"]["verdict"]
                entry["sel_adj_l2"] = data["h1"]["pooled"].get("sel_adj_l2")
            elif key == "provenance":
                entry["commit"] = data.get("git", {}).get("commit")
                entry["dirty"] = data.get("git", {}).get("dirty")
            else:
                entry["nominated"] = [r["feature_key"] for r in data.get("nominated") or []]
        entry["files"], entry["bytes"] = _tree_stats(run_dir)
        manifest["runs"].append(entry)
    (batch_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    target = DRIVE_BATCH / "manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(manifest, indent=2))
    print(f"manifest -> {target}", flush=True)

def build_cmd(task_id, seed, run_dir):
    cmd = [
        str(PYTHON), "-u", "scripts/probe_pi05_transcoder_counterfactual.py",
        "--policy-path", POLICY_PATH,
        "--output-dir", str(run_dir),
        "--suite", SUITE,
        "--task-id", str(task_id),
        "--seed", str(seed),
        "--device", "cuda",
        "--policy-dtype", "bfloat16",
    ]
    if LIST_OBJECTS_ONLY:
        cmd.append("--list-objects")
        return cmd
    cmd += [
        "--checkpoint", TRANSCODER_CHECKPOINT,
        "--perturbation", PERTURBATION,
        "--color", COLOR,
        "--dose", DOSE,
        "--states", str(STATES),
        "--state-stride", str(STATE_STRIDE),
        "--noise-samples", str(NOISE_SAMPLES),
        "--noise-seed", str(seed),
    ]
    # Omitted flags let each task pick its own object of interest and sibling.
    if TARGET.strip():
        cmd += ["--target", TARGET.strip()]
    if PLACEBO_TARGET.strip():
        cmd += ["--placebo-target", PLACEBO_TARGET.strip()]
    return cmd


env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
# Colab's inline matplotlib backend is only valid inside the kernel process;
# inheriting it breaks LIBERO's import. The script guards this too.
env["MPLBACKEND"] = "Agg"
env["MUJOCO_GL"] = "egl"
env["PYOPENGL_PLATFORM"] = "egl"
env["PYTHONPATH"] = str(LOCAL_REPO / "src") + (
    os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else ""
)


def emit(chunk: bytes) -> None:
    """Write through to the cell output.

    Colab's sys.stdout is an ipykernel OutStream with no .buffer, so decode
    rather than assuming a binary stream exists.
    """
    stream = getattr(sys.stdout, "buffer", None)
    if stream is not None:
        stream.write(chunk)
    else:
        sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()


def run_probe(task_id, seed):
    run_dir = batch_dir / f"task{task_id:02d}_seed{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)
    cmd = build_cmd(task_id, seed, run_dir)
    log_path = run_dir / "probe.log"
    print(f"\n{'=' * 78}\n== task {task_id}, seed {seed}\n$ {' '.join(cmd)}\n", flush=True)
    with log_path.open("wb") as log_handle:
        process = subprocess.Popen(cmd, cwd=LOCAL_REPO, env=env,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
        try:
            while True:
                chunk = process.stdout.read(4096)
                if not chunk:
                    break
                log_handle.write(chunk); log_handle.flush()
                emit(chunk)
            rc = process.wait()
        except BaseException:
            # Never leave the probe running headless if the cell is interrupted.
            process.kill(); process.wait(); raise
    return rc, run_dir, log_path


# One task failing must not discard the tasks that already ran: this is an
# hour-long loop and a mid-run abort would cost all of it.
started = time.time()
run_dirs, failures = [], []
plan = [(t, s) for t in TASK_ID_LIST for s in SEED_LIST]
for index, (task_id, seed) in enumerate(plan, start=1):
    print(f"\n[{index}/{len(plan)}]  elapsed {(time.time() - started) / 60:.1f} min", flush=True)
    rc, run_dir, log_path = run_probe(task_id, seed)
    if rc == 0:
        run_dirs.append(run_dir)
        # Immediately, not at the end of the loop: a disconnect twelve runs in
        # would otherwise take all twelve with it.
        save_to_drive(run_dir)
    else:
        failures.append((task_id, seed, rc))
        print(f"\n!! task {task_id} seed {seed} exited {rc}; continuing. Tail of {log_path}:", flush=True)
        for line in log_path.read_text(errors="replace").splitlines()[-12:]:
            print("   ", line[-300:], flush=True)

elapsed = time.time() - started
print(f"\n{'=' * 78}")
print(f"{len(run_dirs)} of {len(plan)} runs completed in {elapsed / 60:.1f} min")
if failures:
    print("failed:", failures)
if not run_dirs:
    raise SystemExit("No task produced a run; nothing to aggregate.")
out_dir = run_dirs[0]  # the trace stage below works from a single task's run

if LIST_OBJECTS_ONLY:
    display(Markdown(
        "### Objects listed\n\nSet `LIST_OBJECTS_ONLY = False` to run the measurement. "
        "Override `TARGET` / `PLACEBO_TARGET` only if you do not want the task's own objects."
    ))
else:
    display(Markdown("### Per-task verdicts"))
    rows = []
    for run_dir in run_dirs:
        try:
            d = json.loads((run_dir / "decision_metrics.json").read_text())
        except FileNotFoundError:
            continue
        h1, h2 = d["h1"], d["h2"]
        rows.append(
            f"<tr><td>{run_dir.name}</td>"
            f"<td>{len(h1['cells'])}</td>"
            f"<td>{h1['pooled']['sel_raw']:.2f}</td>"
            f"<td>{(h1['pooled']['sel_adj_l2'] or float('nan')):.2f}</td>"
            f"<td>{h1['verdict'][:28]}</td>"
            f"<td>{(h2.get('referent_check') or {}).get('referent_moved')}</td>"
            f"<td>{h2['verdict'][:34]}</td></tr>"
        )
    display(HTML(
        "<table><tr><th>run</th><th>cells</th><th>Sel raw</th><th>Sel adj</th>"
        "<th>H1</th><th>referent moved</th><th>H2</th></tr>" + "".join(rows) + "</table>"
    ))
    print("Single-task verdicts are descriptive. The pooled verdict is in the next cell.")

    shapes_path = out_dir / "observation_shapes.json"
    if shapes_path.exists():
        shapes = json.loads(shapes_path.read_text())
        display(Markdown("### Observation Shapes Sent To The Policy"))
        display(HTML(
            "<table><tr><th>key</th><th>shape</th><th>dtype</th></tr>"
            + "".join(f"<tr><td>{k}</td><td>{v['shape']}</td><td>{v['dtype']}</td></tr>"
                      for k, v in sorted(shapes.get("tensor_shapes", {}).items()))
            + f"<tr><td>noise</td><td>{shapes.get('noise_shape')}</td>"
              f"<td>{shapes.get('noise_dtype')}</td></tr></table>"
        ))
        print("task:", repr(shapes.get("task")))

    decision_path = out_dir / "decision_metrics.json"
    if decision_path.exists():
        decision = json.loads(decision_path.read_text())
        h1, h2 = decision["h1"], decision["h2"]

        def _f(v, spec=".4g"):
            return "n/a" if v is None else format(float(v), spec)

        pooled, spread, ci = h1["pooled"], h1["spread"], h1["bootstrap_ci_95"]
        display(Markdown(f"### H1 -- object selectivity (task prompt, {len(h1['cells'])} cells)"))
        display(HTML(
            "<table><tr><th>quantity</th><th>target</th><th>placebo</th><th>ratio</th></tr>"
            f"<tr><td>D, layer response (latent L2)</td><td>{_f(pooled['D_target'])}</td><td>{_f(pooled['D_placebo'])}</td><td>{_f(pooled['sel_raw'])}</td></tr>"
            f"<tr><td>S, changed-pixel fraction</td><td>{_f(pooled['S_px_target'])}</td><td>{_f(pooled['S_px_placebo'])}</td><td>{_f(pooled['S_px_ratio'])}</td></tr>"
            f"<tr><td>S, image-difference norm</td><td>{_f(pooled['S_l2_target'])}</td><td>{_f(pooled['S_l2_placebo'])}</td><td>{_f(pooled['S_l2_ratio'])}</td></tr>"
            f"<tr><td>action relative L2</td><td>{_f(pooled['A_target'])}</td><td>{_f(pooled['A_placebo'])}</td><td>{_f(pooled['action_sel'])}</td></tr>"
            "</table>"
        ))
        display(HTML(
            "<table><tr><th>selectivity</th><th>pooled</th><th>cells mean ± sd</th><th>min .. max</th><th>95% CI</th></tr>"
            + "".join(
                f"<tr><td>{label}</td><td>{_f(pooled[k])}</td><td>{_f(spread[k]['mean'])} ± {_f(spread[k]['sd'])}</td>"
                f"<td>{_f(spread[k]['min'])} .. {_f(spread[k]['max'])}</td><td>[{_f(ci[k]['low'])}, {_f(ci[k]['high'])}]</td></tr>"
                for k, label in (("sel_raw", "raw"), ("sel_adj_px", "adjusted, |P|"), ("sel_adj_l2", "adjusted, ||dI||"))
            )
            + "</table>"
        ))
        nf = h1["null_floor"]
        print(f"null floor D: mean={_f(nf['mean'])} max={_f(nf['max'])} -> {'near zero' if h1['null_floor_near_zero'] else 'NOT near zero'}")
        pc = h1.get("positive_control", {})
        if pc.get("n"):
            print(f"positive control (prompt swap) D={_f(pc['D'])} (n={pc['n']}): target is {_f(pc['target_over_positive'])} of it, "
                  f"placebo {_f(pc['placebo_over_positive'])}")
        cs = h1.get("cluster_spread", {})
        if cs.get("n"):
            sp = cs["sel_adj_l2"]
            print(f"clusters {cs['unit']}: n={cs['n']}  Sel adj ||dI|| min={_f(sp['min'])} max={_f(sp['max'])}")
        rb = h1.get("robustness", {})
        if rb:
            print(f"robustness: Sel rel={_f(rb['sel_rel'])}  per-layer geomean={_f(rb['sel_layer_geomean'])} "
                  f"[{_f(rb['sel_layer_min'])} .. {_f(rb['sel_layer_max'])}], {rb['layers_above_one']}/{rb['layers']} layers above 1; "
                  f"direction agrees: {rb['agree_in_direction']}")
        for kind, dr in h1["dose_response"].items():
            print(f"dose response [{kind}]: elasticity={_f(dr['elasticity_mean'])} monotone={dr['monotone_in_dose']}  "
                  + "  ".join(f"d={r['dose']:g}: D={_f(r['D'])}" for r in dr["rows"]))
        display(Markdown(f"**H1 verdict: {h1['verdict']}**  \n<small>rule: {h1['decision_rule']}</small>"))

        g = h2["grounding"]
        display(Markdown("### H2 -- referent versus position (prompt swap over identical pixels)"))
        display(HTML(
            "<table><tr><th>prompt</th><th>D target</th><th>D placebo</th><th>selectivity</th><th>cells</th></tr>"
            + "".join(
                f"<tr><td>{p}</td><td>{_f(r['D_target'])}</td><td>{_f(r['D_placebo'])}</td><td>{_f(r['sel_raw'])}</td><td>{r['n_cells']}</td></tr>"
                for p, r in h2["per_prompt"].items()
            )
            + "</table>"
        ))
        print(f"prompt grounding g: mean={_f(g['mean'])} min={_f(g['min'])} max={_f(g['max'])} (n={g['n']}, threshold {h2['grounding_threshold']}); "
              f"perturbation action effect {_f(h2['perturbation_action_effect'])}")
        rc = h2.get("referent_check", {})
        if rc:
            print(f"referent check (action anchor A_target/A_placebo): task={_f(rc.get('behavioural_anchor_task'))} "
                  f"alt={_f(rc.get('behavioural_anchor_alt'))} -> referent moved: {rc.get('referent_moved')}")
        display(Markdown(f"**H2 verdict: {h2['verdict']}**  \n<small>rule: {h2['decision_rule']}</small>"))

    provenance_path = out_dir / "provenance.json"
    if provenance_path.exists():
        pv = json.loads(provenance_path.read_text())
        ck = (pv.get("transcoder_checkpoint") or {}).get("digest", "")
        print(f"provenance: commit {pv['git'].get('commit')} dirty={pv['git'].get('dirty')} "
              f"lerobot={pv['packages'].get('lerobot')} torch={pv['packages'].get('torch')} "
              f"device={pv['device'].get('gpu_name', pv['device'].get('device'))} checkpoint sha256={ck[:16]}")

    summary_path = out_dir / "counterfactual_summary.json"
    if summary_path.exists():
        summary = json.loads(summary_path.read_text())
        display(Markdown("### Counterfactual Activation Response (every measurement)"))
        display(HTML(
            "<table><tr><th>state</th><th>noise</th><th>prompt</th><th>kind</th><th>target</th><th>dose</th>"
            "<th>changed pixels</th><th>latent L2</th><th>action rel L2</th></tr>"
            + "".join(
                "<tr><td>{s}</td><td>{n}</td><td>{p}</td><td>{k}</td><td>{t}</td><td>{d}</td><td>{px:.5f}</td>"
                "<td>{lat}</td><td>{act:.5g}</td></tr>".format(
                    s=r["state_index"], n=r.get("noise_index", 0), p=r.get("prompt", "task"),
                    k=r["kind"], t=r["target"], d=r["dose"],
                    px=r["pixel"]["changed_pixel_fraction"],
                    lat=("n/a" if r["latent"].get("l2_delta_mean") is None
                         else "{:.5g}".format(r["latent"]["l2_delta_mean"])),
                    act=r["action_relative_l2"])
                for r in summary.get("measurements", []))
            + "</table>"
        ))
        print("Read every row against the 'null' rows: that is the nondeterminism floor.")

    # One state's frames for the eye: every camera's baseline, the target and
    # placebo recolours at the top dose, and their difference images.
    pngs = sorted((out_dir / "images").glob("state0_*.png"))
    keep = [p for p in pngs if "baseline" in p.name or "dose1_" in p.name]
    for png in keep[:16]:
        display(Markdown(f"**{png.name}**"))
        display(IPyImage(filename=str(png)))

    # Rank features by selectivity: responds to the task object, not to the
    # visually matched control.
    report_cmd = [str(PYTHON), "-u", "scripts/report_pi05_counterfactual_features.py",
                  str(out_dir), "--top", "30",
                  "--min-consistency", str(MIN_TRACE_CONSISTENCY),
                  "--min-trace-layer", str(MIN_TRACE_LAYER),
                  "--min-selectivity", str(MIN_SELECTIVITY)]
    print("\n$", " ".join(report_cmd), flush=True)
    report = subprocess.run(report_cmd, cwd=LOCAL_REPO, env=env, capture_output=True, text=True)
    print(report.stdout)
    if report.returncode != 0:
        print(report.stderr[-2000:])

# Every run was copied as it finished; this re-copies whatever the report stage
# added afterwards (candidate_features.csv, nominated_targets.json) and writes
# the batch index.
for run_dir in run_dirs:
    save_to_drive(run_dir)
write_manifest()
print("\nArtifacts:", batch_dir)
if SAVE_RUNS_TO_DRIVE:
    print("On Drive:", DRIVE_BATCH)


## Pool the tasks

The single-task run decided H1 on the minimum adjusted selectivity over its
eight cells, which was a crutch for having four independent clusters. Cells are
not independent: doses within a (state, draw) share a render and a noise sample,
draws within a state share a pose, and states within a task share a scene. The
task is the exchangeable unit, so it is what the interval resamples.

Read three things here. The **pooled adjusted selectivity with its task-cluster
interval**, which is the H1 verdict. The **per-task column**, which shows whether
the effect is uniform or carried by a few tasks. And the **H2 table**, where each
task either moved its behavioural referent or did not; only the ones that did
can be read, and the pilot's single task did not.


In [ ]:
# @title Aggregate Across Tasks

import json, os, subprocess, sys
from pathlib import Path
from IPython.display import display, HTML, Markdown

agg_dir = batch_dir / "aggregate"
cmd = [str(PYTHON), "-u", "scripts/aggregate_counterfactual_runs.py", *[str(d) for d in run_dirs],
       "--output-dir", str(agg_dir)]
print("$", " ".join(cmd), flush=True)
proc = subprocess.run(cmd, cwd=LOCAL_REPO, env=env, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr[-3000:]); raise SystemExit(f"aggregation exited {proc.returncode}")

report = json.loads((agg_dir / "aggregate.json").read_text())
s = report["summary"]

def _f(v, spec=".2f"):
    return "n/a" if v is None else format(float(v), spec)

display(Markdown(f"### H1 pooled over {s['tasks']} tasks ({s['cells_total']} cells)"))
p, ci = s["pooled"], s["bootstrap_ci_95"]
display(HTML(
    "<table><tr><th>quantity</th><th>pooled</th><th>task-cluster 95% CI</th></tr>"
    + "".join(
        f"<tr><td>{label}</td><td>{_f(p.get(k))}</td>"
        f"<td>[{_f(ci[k].get('low'))}, {_f(ci[k].get('high'))}]</td></tr>"
        for k, label in (("sel_raw", "raw selectivity"),
                         ("sel_adj_l2", "adjusted, image norm (decides)"),
                         ("sel_adj_px", "adjusted, pixel count")))
    + "</table>"
))
print(f"tasks with adjusted selectivity above 1: {s['tasks_above_one']} of {s['tasks']}")

display(Markdown("**Per task** (the spread here is what the pilot could not see)"))
display(HTML(
    "<table><tr><th>task</th><th>cells</th><th>Sel raw</th><th>Sel adj</th><th>action sel</th></tr>"
    + "".join(
        f"<tr><td>{r['task_id']}</td><td>{r['n_cells']}</td><td>{_f(r['sel_raw'])}</td>"
        f"<td>{_f(r['sel_adj_l2'])}</td><td>{_f(r['action_sel'])}</td></tr>"
        for r in report["per_task"])
    + "</table>"
))
display(Markdown(f"**H1 verdict: {s['verdict']}**  \n<small>rule: {s['decision_rule']}</small>"))

display(Markdown("### H2, gated per task"))
display(HTML(
    "<table><tr><th>task</th><th>g</th><th>anchor task</th><th>anchor alt</th>"
    "<th>referent moved</th><th>Sel task</th><th>Sel alt</th></tr>"
    + "".join(
        f"<tr><td>{r['task_id']}</td><td>{_f(r['grounding'],'.3f')}</td><td>{_f(r['anchor_task'])}</td>"
        f"<td>{_f(r['anchor_alt'])}</td><td><b>{r['referent_moved']}</b></td>"
        f"<td>{_f(r['sel_task'])}</td><td>{_f(r['sel_alt'])}</td></tr>"
        for r in report["h2_per_task"])
    + "</table>"
))
display(Markdown(f"**H2 verdict: {s['h2_verdict']}**"))

save_to_drive(agg_dir)
print("\nPaste the two verdict lines and the per-task tables into the write-up.")


## Trace and audit, as a rate rather than an anecdote

The pilot traced one nominated target, found its parents no more object-selective
than random, and recorded H3 as falsified. Two things were missing.

**A positive control for the audit.** The audit had a matched-random negative
control and nothing establishing that it could detect a content-carrying circuit
if one existed, so a flat reading was ambiguous between "this circuit carries
nothing" and "this test is blind". The calibration below builds circuits whose
content is known by construction, from the probe's own measurements, and sweeps
the fraction of genuinely selective members from none to all. The level at which
detection becomes reliable converts a negative from "not enriched" into "less
than this much content-carrying".

**More than one target.** One trace can only ever produce one falsification.
This traces and audits the top `TRACE_N_TARGETS` nominees and reports how many
were corroborated.

It also switches the frontier to previous-layer sources. In the pilot, with
sources drawn from every earlier layer, 45 of 60 expansions collapsed into layer
1 and the graph reached 1681 parents. Whether that caused the flat profile or
merely accompanied it is exactly what this controls for.


In [ ]:
# @title Trace And Audit Several Targets

import csv, json, math, os, subprocess, sys, time
from pathlib import Path
from IPython.display import display, HTML, Markdown

if not RUN_CIRCUIT_TRACE:
    print("RUN_CIRCUIT_TRACE is off; skipping.")
else:
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"; env["MPLBACKEND"] = "Agg"
    env["MUJOCO_GL"] = "egl"; env["PYOPENGL_PLATFORM"] = "egl"
    env["PYTHONPATH"] = str(LOCAL_REPO / "src") + (
        os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")

    def stream(label, cmd, log_path):
        print(f"\n== {label} ==\n$ {' '.join(map(str, cmd))}", flush=True)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        with log_path.open("wb") as handle:
            proc = subprocess.Popen([str(c) for c in cmd], cwd=LOCAL_REPO, env=env,
                                    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
            try:
                while True:
                    chunk = proc.stdout.read(4096)
                    if not chunk:
                        break
                    handle.write(chunk); handle.flush()
                    out = getattr(sys.stdout, "buffer", None)
                    (out.write(chunk) if out is not None else
                     sys.stdout.write(chunk.decode("utf-8", "replace")))
                    sys.stdout.flush()
                return proc.wait()
            except BaseException:
                proc.kill(); proc.wait(); raise

    # --- Pool nominees across every task -------------------------------------
    # A rate computed from one task's nominees is a rate over that scene. Each
    # nominee keeps the run it came from, because its audit has to read the
    # delta store that measured it.
    pool, seen = [], set()
    for run_dir in run_dirs:
        path = run_dir / "nominated_targets.json"
        if not path.exists():
            continue
        for row in json.loads(path.read_text()).get("nominated") or []:
            key = row["feature_key"]
            if key in seen:      # the same feature nominated by two tasks is
                continue         # one circuit, and would be traced once anyway
            seen.add(key)
            pool.append({"run_dir": run_dir, **row})
    if not pool:
        raise RuntimeError(
            "No task nominated a target under the rule. Widen the probe (STATES, SEEDS, DOSE) "
            "or relax MIN_TRACE_CONSISTENCY / MIN_TRACE_LAYER / MIN_SELECTIVITY deliberately."
        )
    pool.sort(key=lambda r: -float(r["target_z"]))
    targets = pool[:TRACE_N_TARGETS]
    bound = 1 - 0.05 ** (1 / len(targets))
    print(f"{len(pool)} distinct nominees across {len(run_dirs)} runs; tracing the top {len(targets)}.")
    print(f"A null result over {len(targets)} bounds the corroboration rate below {bound:.1%}.\n")
    for row in targets:
        print(f"  {row['feature_key']:<22} z={float(row['target_z']):+.2f}  "
              f"sel={float(row['selectivity']):.1f}x  from {row['run_dir'].name}")

    batch_out = batch_dir / "h3"
    batch_out.mkdir(parents=True, exist_ok=True)

    # --- Discovery once, shared by every trace -------------------------------
    stamp = time.strftime("%Y%m%d-%H%M%S", time.gmtime())
    feature_dir = LOCAL_REPO / "outputs/features/pi05_libero" / f"counterfactual-{stamp}"
    rc = stream("Feature discovery", [
        PYTHON, "-u", "scripts/collect_pi05_transcoder_features.py",
        "--checkpoint", TRANSCODER_CHECKPOINT, "--output-dir", feature_dir,
        "--episodes", DISCOVERY_EPISODES, "--max-batches", str(DISCOVERY_MAX_BATCHES),
        "--device", "cuda", "--policy-dtype", "bfloat16"], feature_dir / "discovery.log")
    if rc != 0:
        raise RuntimeError(f"feature discovery exited {rc}")
    save_to_drive(feature_dir)

    # --- Calibrate the audit before trusting any verdict it gives ------------
    # Without this a flat reading cannot be told apart from a blind test. One
    # calibration per (run, target layer), because the pool it samples is that
    # run's own exercised features below that layer.
    calibrations, cal_by_key = {}, {}
    if RUN_AUDIT_CALIBRATION:
        for row in targets:
            layer = int(row["layer_index"])
            ckey = (str(row["run_dir"]), layer)
            cal_by_key[row["feature_key"]] = ckey
            if ckey in calibrations:
                continue
            cal_dir = batch_out / f"calibration_{row['run_dir'].name}_L{layer}"
            rc = stream(f"Audit calibration below layer {layer} ({row['run_dir'].name})", [
                PYTHON, "-u", "scripts/calibrate_circuit_audit.py", row["run_dir"],
                "--target-layer", str(layer), "--tau", str(float(row["tau"])),
                "--size", "16", "--repeats", "20", "--draws", str(VALIDATION_RANDOM_DRAWS),
                "--output-dir", cal_dir], cal_dir / "calibration.log")
            path = cal_dir / "audit_calibration.json"
            calibrations[ckey] = json.loads(path.read_text()) if (rc == 0 and path.exists()) else None
            save_to_drive(cal_dir)

    # --- Trace and audit each target -----------------------------------------
    # Twenty traces is several hours. The summary is rerun after every target
    # and copied to Drive, so a dropped runtime leaves a readable partial
    # result rather than a directory of logs.
    cal_args = []
    for (run_key, layer), cal in calibrations.items():
        if cal:
            cal_args += ["--calibration",
                         str(batch_out / f"calibration_{Path(run_key).name}_L{layer}")]

    def summarize(paths, *, quiet=True):
        rc = stream("Summarise H3", [
            PYTHON, "-u", "scripts/summarize_h3_rate.py", *[str(q) for q in paths], *cal_args,
            "--source-policy", TRACE_SOURCE_POLICY, "--output-dir", batch_out],
            batch_out / "summarize.log")
        if rc != 0:
            raise RuntimeError(f"summarize_h3_rate exited {rc}")
        return json.loads((batch_out / "h3_rate.json").read_text())

    traced, h3 = [], None
    for n, row in enumerate(targets, 1):
        key, run_dir = row["feature_key"], row["run_dir"]
        trace_dir = run_dir / "circuits" / key.replace(":", "_")
        print(f"\n########## {n}/{len(targets)}  {key}  ({run_dir.name}) ##########", flush=True)
        rc = stream(f"Trace {key}", [
            PYTHON, "-u", "scripts/trace_pi05_transcoder_circuit.py",
            "--checkpoint", TRANSCODER_CHECKPOINT, "--feature-dir", feature_dir,
            "--output-dir", trace_dir, "--target", key,
            "--source-policy", TRACE_SOURCE_POLICY,
            "--parents-per-node", str(TRACE_PARENTS_PER_NODE),
            "--max-depth", str(TRACE_MAX_DEPTH), "--max-nodes", str(TRACE_MAX_NODES),
            "--device", "cuda", "--policy-dtype", "bfloat16"], trace_dir / "trace.log")
        if rc == 0:
            rc = stream(f"Audit {key}", [
                PYTHON, "-u", "scripts/validate_circuit_with_counterfactual.py",
                trace_dir, run_dir, "--random-draws", str(VALIDATION_RANDOM_DRAWS)],
                trace_dir / "validation.log")
        else:
            print(f"  trace exited {rc}; the feature may not fire in the discovery pool.")
        # Saved either way: a failed trace's log is how the failure gets diagnosed,
        # and the summary counts it as failed rather than dropping it.
        save_to_drive(trace_dir, required=False)
        traced.append(trace_dir)
        h3 = summarize(traced)
        running = h3["summary"]
        note = f", {running['failed']} failed" if running["failed"] else ""
        print(f"  running total: {running['corroborated']}/{running['audited']} "
              f"corroborated{note}", flush=True)
        save_to_drive(batch_out, required=False)

    # --- Report ----------------------------------------------------------------
    # The rate, its interval and the verdict come from a script with its own
    # tests and are read back from the artefact. Nothing here re-derives them.
    summary, audits = h3["summary"], h3["audits"]

    def _f(x, spec=".3f"):
        return "n/a" if x is None else format(float(x), spec)

    if calibrations:
        display(Markdown("### What the audit can detect"))
        rows = ""
        for ckey, cal in calibrations.items():
            if not cal:
                rows += f"<tr><td colspan=4><i>calibration failed for {Path(ckey[0]).name} L{ckey[1]}</i></td></tr>"
                continue
            for r in cal["levels"]:
                rows += (f"<tr><td>{Path(ckey[0]).name} L{ckey[1]}</td>"
                         f"<td>{r['contamination']:.0%}</td>"
                         f"<td>{'inf' if r['enrichment_median'] is None else format(r['enrichment_median'], '.2f')}</td>"
                         f"<td>{r['detection_rate']:.0%}</td></tr>")
        display(HTML("<table><tr><th>pool</th><th>content</th><th>sel enrich (median)</th>"
                     "<th>detected</th></tr>" + rows + "</table>"))
        if summary["blind_pools"]:
            display(Markdown(f"**The audit is blind on {len(summary['blind_pools'])} pool(s); "
                             "no verdict below means anything.**"))
        elif summary["detection_floor"] is not None:
            display(Markdown(
                f"**Detection floor: the audit finds a set that is at least "
                f"{summary['detection_floor']:.0%} content-carrying. "
                f"A circuit it reads as flat is below that.**"))

    display(Markdown(f"### H3 over {summary['audited']} audited circuits"))
    sel_by_key = {r["feature_key"]: float(r["selectivity"]) for r in targets}
    task_by_key = {r["feature_key"]: r["run_dir"].name for r in targets}
    display(HTML(
        "<table><tr><th>target</th><th>task</th><th>feature sel</th><th>parents</th>"
        "<th>exercised</th><th>head n</th><th>head sel enrich</th><th>p</th>"
        "<th>full</th><th>spec</th><th>verdict</th></tr>"
        + "".join(
            f"<tr><td>{r.get('target') or r['path'][-30:]}</td>"
            f"<td>{task_by_key.get(r.get('target'), '')}</td>"
            f"<td>{_f(sel_by_key.get(r.get('target')), '.1f')}</td><td>{r.get('parents') or ''}</td>"
            f"<td>{_f(r.get('exercised'), '.1%')}</td><td>{r.get('head_nodes') or ''}</td>"
            f"<td>{_f(r.get('head_enrichment'))}</td><td>{_f(r.get('head_p'), '.4f')}</td>"
            f"<td>{_f(r.get('full_enrichment'))}</td><td>{_f(r.get('specificity'), '.2f')}</td>"
            f"<td>{str(r.get('verdict') or r['status'])[:38]}</td></tr>" for r in audits)
        + "</table>"
    ))
    ci = summary["clopper_pearson_95"]
    if summary["audited"]:
        line = (f"**H3: {summary['corroborated']} of {summary['audited']} circuits corroborated "
                f"= {summary['rate']:.0%}, 95% CI [{ci['low']:.1%}, {ci['high']:.1%}]**")
        if summary["reading"]:
            line += f"\n\n{summary['reading']}."
        display(Markdown(line))
    display(Markdown(f"**Verdict: {summary['verdict']}**"))

    save_to_drive(batch_out)
    write_manifest(h3=summary)
    print(f"\nH3 artefacts: {batch_out}")
    if SAVE_RUNS_TO_DRIVE:
        print("On Drive:", DRIVE_BATCH / "h3")
    print("Paste the detection floor and the H3 rate into the write-up.")
